In [1]:
import json
from pathlib import Path

import pandas as pd

In [2]:
# Input and output directories
ROOT_DIR = Path("ctg_full_2005_2025")
COHORT_DIR = ROOT_DIR / "cohort_qc"
OUTPUT_DIR = ROOT_DIR / "disease_grouping"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# Input file
ANALYTIC_COHORT_FILE = (
    COHORT_DIR / "analytic_cohort.csv"
)


# Disease-domain classification outputs
CLASSIFIED_COHORT_FILE = (
    OUTPUT_DIR / "analytic_cohort_disease_classified.csv"
)

DISEASE_ANALYSIS_COHORT_FILE = (
    OUTPUT_DIR / "disease_analysis_cohort.csv"
)

DISEASE_GROUP_SUMMARY_FILE = (
    OUTPUT_DIR / "disease_group_summary.csv"
)

CLASSIFICATION_STATUS_FILE = (
    OUTPUT_DIR / "disease_classification_status_summary.csv"
)

OVERLAP_SUMMARY_FILE = (
    OUTPUT_DIR / "disease_domain_overlap_summary.csv"
)

In [3]:
# Selected disease domains and corresponding MeSH concepts
DISEASE_DOMAINS = {
    "Neoplasms": "D009369",
    "Nervous System Diseases": "D009422",
    "Cardiovascular Diseases": "D002318",
    "Metabolic Diseases": "D008659",
    "Respiratory Tract Diseases": "D012140",
    "Infections": "D007239",
}

In [4]:
def parse_json_list(value):
    """Parse a JSON-encoded list stored in the analytic cohort CSV."""

    if pd.isna(value):
        return []

    parsed = json.loads(value)

    if not isinstance(parsed, list):
        raise ValueError(
            "Expected a JSON-encoded list."
        )

    return parsed


def get_study_mesh_ids(
    condition_meshes,
    condition_ancestors,
):
    """
    Return the union of direct and ancestor condition MeSH IDs
    for one study.

    Every list element must be a dictionary containing a non-empty
    MeSH ID. Unexpected records raise an error rather than being
    silently ignored.
    """

    def extract_ids(items, field_name):
        ids = set()

        for item in items:
            if not isinstance(item, dict):
                raise ValueError(
                    f"Unexpected item in {field_name}: "
                    "expected a dictionary."
                )

            mesh_id = item.get("id")

            if not isinstance(mesh_id, str) or not mesh_id.strip():
                raise ValueError(
                    f"Missing or invalid MeSH ID in {field_name}."
                )

            ids.add(mesh_id.strip())

        return ids

    direct_ids = extract_ids(
        condition_meshes,
        "condition_meshes",
    )

    ancestor_ids = extract_ids(
        condition_ancestors,
        "condition_ancestors",
    )

    return direct_ids | ancestor_ids


def get_disease_domains(mesh_ids):
    """Return selected disease domains represented in one study."""

    return [
        domain
        for domain, mesh_id in DISEASE_DOMAINS.items()
        if mesh_id in mesh_ids
    ]


def get_classification_status(
    mesh_ids,
    disease_domains,
):
    """Assign disease-classification status for one study."""

    if not mesh_ids:
        return "no_mesh"

    if len(disease_domains) == 0:
        return "other_domain"

    if len(disease_domains) == 1:
        return "single_domain"

    return "multiple_domains"

In [5]:
# Load analytic cohort
if not ANALYTIC_COHORT_FILE.exists():
    raise RuntimeError(
        f"Analytic cohort not found: "
        f"{ANALYTIC_COHORT_FILE}"
    )

df = pd.read_csv(
    ANALYTIC_COHORT_FILE,
    encoding="utf-8-sig",
)


required_columns = [
    "nct_id",
    "condition_meshes",
    "condition_ancestors",
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise RuntimeError(
        "Required columns are missing from the analytic cohort: "
        f"{missing_columns}"
    )


n_records = len(df)

n_unique_ids = df["nct_id"].nunique(
    dropna=True
)

n_missing_ids = int(
    df["nct_id"].isna().sum()
)

n_duplicate_ids = int(
    df["nct_id"]
    .duplicated(keep=False)
    .sum()
)


if n_missing_ids != 0:
    raise RuntimeError(
        f"Missing NCT IDs found: {n_missing_ids}"
    )

if n_duplicate_ids != 0:
    raise RuntimeError(
        f"Duplicate NCT-ID rows found: {n_duplicate_ids}"
    )

if n_unique_ids != n_records:
    raise RuntimeError(
        "The number of unique NCT IDs does not equal "
        "the number of study records."
    )


print(f"Analytic cohort records: {n_records}")
print(f"Unique NCT IDs: {n_unique_ids}")

Analytic cohort records: 405875
Unique NCT IDs: 405875


In [6]:
# Parse MeSH fields
df["condition_meshes_parsed"] = (
    df["condition_meshes"]
    .apply(parse_json_list)
)

df["condition_ancestors_parsed"] = (
    df["condition_ancestors"]
    .apply(parse_json_list)
)

In [7]:
# Combine direct and ancestor MeSH IDs
df["condition_mesh_ids"] = [
    get_study_mesh_ids(
        condition_meshes,
        condition_ancestors,
    )
    for condition_meshes, condition_ancestors
    in zip(
        df["condition_meshes_parsed"],
        df["condition_ancestors_parsed"],
    )
]


# Identify all selected disease domains represented in each study
df["disease_domains"] = (
    df["condition_mesh_ids"]
    .apply(get_disease_domains)
)

df["n_disease_domains"] = (
    df["disease_domains"]
    .apply(len)
)


# Create non-mutually-exclusive disease-domain indicators
for domain in DISEASE_DOMAINS:

    column_name = {
        "Neoplasms": "domain_neoplasms",
        "Nervous System Diseases": "domain_nervous_system_diseases",
        "Cardiovascular Diseases": "domain_cardiovascular_diseases",
        "Metabolic Diseases": "domain_metabolic_diseases",
        "Respiratory Tract Diseases": "domain_respiratory_tract_diseases",
        "Infections": "domain_infections",
    }[domain]

    df[column_name] = (
        df["disease_domains"]
        .apply(lambda domains: domain in domains)
    )


# Assign classification status for QC and reporting
df["disease_classification_status"] = [
    get_classification_status(
        mesh_ids,
        disease_domains,
    )
    for mesh_ids, disease_domains
    in zip(
        df["condition_mesh_ids"],
        df["disease_domains"],
    )
]

In [8]:
# Disease-classification status
classification_status_df = (
    df["disease_classification_status"]
    .value_counts()
    .rename_axis("classification_status")
    .reset_index(name="n_studies")
)

classification_status_df["percent"] = (
    classification_status_df["n_studies"]
    / len(df)
    * 100
)

classification_status_df.to_csv(
    CLASSIFICATION_STATUS_FILE,
    index=False,
    encoding="utf-8-sig",
)


print("Disease-classification status:")
print(
    classification_status_df.to_string(
        index=False
    )
)

Disease-classification status:
classification_status  n_studies   percent
        single_domain     153677 37.863135
         other_domain     120299 29.639421
              no_mesh      87905 21.658146
     multiple_domains      43994 10.839298


In [9]:
# Disease-domain membership summary
domain_summary_rows = []

for domain in DISEASE_DOMAINS:

    column_name = {
        "Neoplasms": "domain_neoplasms",
        "Nervous System Diseases": "domain_nervous_system_diseases",
        "Cardiovascular Diseases": "domain_cardiovascular_diseases",
        "Metabolic Diseases": "domain_metabolic_diseases",
        "Respiratory Tract Diseases": "domain_respiratory_tract_diseases",
        "Infections": "domain_infections",
    }[domain]

    n_studies = int(
        df[column_name].sum()
    )

    domain_summary_rows.append(
        {
            "disease_domain": domain,
            "n_studies": n_studies,
            "percent_full_cohort": (
                n_studies / len(df) * 100
            ),
        }
    )


disease_group_summary_df = pd.DataFrame(
    domain_summary_rows
)


disease_group_summary_df.to_csv(
    DISEASE_GROUP_SUMMARY_FILE,
    index=False,
    encoding="utf-8-sig",
)


print()
print("Disease-domain membership summary:")
print(
    disease_group_summary_df.to_string(
        index=False
    )
)


Disease-domain membership summary:
            disease_domain  n_studies  percent_full_cohort
                 Neoplasms      70155            17.284878
   Nervous System Diseases      49429            12.178380
   Cardiovascular Diseases      38373             9.454389
        Metabolic Diseases      25360             6.248229
Respiratory Tract Diseases      33028             8.137481
                Infections      28510             7.024330


In [10]:
# Multi-domain membership combinations
overlap_summary_df = (
    df.loc[
        df["n_disease_domains"] > 1,
        "disease_domains",
    ]
    .apply(
        lambda domains: " + ".join(
            sorted(domains)
        )
    )
    .value_counts()
    .rename_axis("domain_combination")
    .reset_index(name="n_studies")
)

overlap_summary_df.to_csv(
    OVERLAP_SUMMARY_FILE,
    index=False,
    encoding="utf-8-sig",
)


print()
print("Most common multi-domain membership combinations:")
print(
    overlap_summary_df
    .head(20)
    .to_string(index=False)
)


Most common multi-domain membership combinations:
                                                            domain_combination  n_studies
                                       Infections + Respiratory Tract Diseases       8734
                             Cardiovascular Diseases + Nervous System Diseases       7962
                                        Neoplasms + Respiratory Tract Diseases       7642
                                           Neoplasms + Nervous System Diseases       3177
                                           Cardiovascular Diseases + Neoplasms       2817
                                  Cardiovascular Diseases + Metabolic Diseases       2345
                                  Metabolic Diseases + Nervous System Diseases       2172
                          Nervous System Diseases + Respiratory Tract Diseases       1891
                          Cardiovascular Diseases + Respiratory Tract Diseases       1227
                                                 

In [11]:
# Classification reconciliation
n_single_domain = int(
    (
        df["disease_classification_status"]
        == "single_domain"
    ).sum()
)

n_other_domain = int(
    (
        df["disease_classification_status"]
        == "other_domain"
    ).sum()
)

n_no_mesh = int(
    (
        df["disease_classification_status"]
        == "no_mesh"
    ).sum()
)

n_multiple_domains = int(
    (
        df["disease_classification_status"]
        == "multiple_domains"
    ).sum()
)


if (
    n_single_domain
    + n_other_domain
    + n_no_mesh
    + n_multiple_domains
    != len(df)
):
    raise RuntimeError(
        "Disease-classification counts do not reconcile "
        "to the analytic cohort."
    )


# Classification status must agree with the number of selected domains.
if not (
    df.loc[
        df["disease_classification_status"] == "single_domain",
        "n_disease_domains",
    ]
    == 1
).all():
    raise RuntimeError(
        "Single-domain studies do not all have exactly "
        "one selected disease domain."
    )


if not (
    df.loc[
        df["disease_classification_status"] == "multiple_domains",
        "n_disease_domains",
    ]
    > 1
).all():
    raise RuntimeError(
        "Multiple-domain studies do not all have more than "
        "one selected disease domain."
    )


if not (
    df.loc[
        df["disease_classification_status"].isin(
            ["no_mesh", "other_domain"]
        ),
        "n_disease_domains",
    ]
    == 0
).all():
    raise RuntimeError(
        "Studies outside the selected disease domains "
        "have non-zero domain membership."
    )


# Boolean disease-domain indicators must reconcile with
# n_disease_domains.
domain_indicator_columns = [
    "domain_neoplasms",
    "domain_nervous_system_diseases",
    "domain_cardiovascular_diseases",
    "domain_metabolic_diseases",
    "domain_respiratory_tract_diseases",
    "domain_infections",
]

indicator_domain_count = (
    df[domain_indicator_columns]
    .sum(axis=1)
)

if not (
    indicator_domain_count
    == df["n_disease_domains"]
).all():
    raise RuntimeError(
        "Disease-domain indicators do not reconcile with "
        "n_disease_domains."
    )


print("Disease-classification integrity checks passed.")
print(f"Single-domain studies: {n_single_domain}")
print(f"Other-domain studies: {n_other_domain}")
print(f"Studies without MeSH classification: {n_no_mesh}")
print(f"Multiple-domain studies: {n_multiple_domains}")

Disease-classification integrity checks passed.
Single-domain studies: 153677
Other-domain studies: 120299
Studies without MeSH classification: 87905
Multiple-domain studies: 43994


In [12]:
# Remove temporary working columns
classified_cohort_df = df.drop(
    columns=[
        "condition_meshes_parsed",
        "condition_ancestors_parsed",
        "condition_mesh_ids",
    ]
).copy()


# Serialize disease-domain membership only in the output dataframe.
classified_cohort_df["disease_domains"] = (
    classified_cohort_df["disease_domains"]
    .apply(
        lambda domains: json.dumps(
            domains,
            ensure_ascii=False,
        )
    )
)

In [13]:
# Save full classified analytic cohort
classified_cohort_df.to_csv(
    CLASSIFIED_COHORT_FILE,
    index=False,
    encoding="utf-8-sig",
)

print(
    f"Classified analytic cohort saved to: "
    f"{CLASSIFIED_COHORT_FILE}"
)
print(
    f"Studies saved: "
    f"{len(classified_cohort_df)}"
)

Classified analytic cohort saved to: ctg_full_2005_2025/disease_grouping/analytic_cohort_disease_classified.csv
Studies saved: 405875


In [14]:
# Create disease-analysis cohort.
#
# Retain studies belonging to at least one selected disease domain.
# Disease-domain membership is non-mutually exclusive, and each study
# remains represented by one row.
disease_analysis_df = (
    classified_cohort_df.loc[
        classified_cohort_df[
            "n_disease_domains"
        ] >= 1
    ]
    .copy()
    .reset_index(drop=True)
)


if (
    disease_analysis_df["nct_id"].nunique()
    != len(disease_analysis_df)
):
    raise RuntimeError(
        "Disease-analysis cohort contains duplicate NCT IDs."
    )


if (
    disease_analysis_df["n_disease_domains"] < 1
).any():
    raise RuntimeError(
        "Disease-analysis cohort contains studies with "
        "no selected disease-domain membership."
    )


domain_indicator_columns = [
    "domain_neoplasms",
    "domain_nervous_system_diseases",
    "domain_cardiovascular_diseases",
    "domain_metabolic_diseases",
    "domain_respiratory_tract_diseases",
    "domain_infections",
]

if not (
    disease_analysis_df[
        domain_indicator_columns
    ].sum(axis=1)
    == disease_analysis_df["n_disease_domains"]
).all():
    raise RuntimeError(
        "Disease-domain indicators do not reconcile with "
        "n_disease_domains in the disease-analysis cohort."
    )


disease_analysis_df.to_csv(
    DISEASE_ANALYSIS_COHORT_FILE,
    index=False,
    encoding="utf-8-sig",
)


print()
print(
    f"Disease-analysis cohort saved to: "
    f"{DISEASE_ANALYSIS_COHORT_FILE}"
)
print(
    f"Studies: "
    f"{len(disease_analysis_df)}"
)
print(
    f"Unique NCT IDs: "
    f"{disease_analysis_df['nct_id'].nunique()}"
)


Disease-analysis cohort saved to: ctg_full_2005_2025/disease_grouping/disease_analysis_cohort.csv
Studies: 197671
Unique NCT IDs: 197671
